In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Create a geocoding script as a Jupyter notebook
notebook_content = """\
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Geocode Toledo Property Addresses\\n",
    "This notebook geocodes a CSV of delinquent property addresses using Nominatim (OpenStreetMap)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\\n",
    "from geopy.geocoders import Nominatim\\n",
    "from geopy.extra.rate_limiter import RateLimiter\\n",
    "import time"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the cleaned address data\\n",
    "df = pd.read_csv(\"cleaned_toledo_addresses.csv\")\\n",
    "df[\"full_address\"] = df[\"street_number\"] + \" \" + df[\"street_name\"] + \", \" + df[\"city\"] + \", \" + df[\"state\"]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize geocoder\\n",
    "geolocator = Nominatim(user_agent=\"toledo_delinquent_properties_geocoder\")\\n",
    "geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Safe geocoding function\\n",
    "def safe_geocode(address):\\n",
    "    try:\\n",
    "        loc = geocode(address)\\n",
    "        return (loc.latitude, loc.longitude) if loc else (None, None)\\n",
    "    except:\\n",
    "        return (None, None)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Apply geocoding\\n",
    "df[[\"latitude\", \"longitude\"]] = df[\"full_address\"].apply(lambda x: pd.Series(safe_geocode(x)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the geocoded data\\n",
    "df.to_csv(\"geocoded_toledo_addresses.csv\", index=False)\\n",
    "print(\"Saved to geocoded_toledo_addresses.csv\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
"""

# Save the notebook to a .ipynb file
notebook_path = "geocode_toledo_addresses.ipynb"
with open(notebook_path, "w") as f:
    f.write(notebook_content)

notebook_path


In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
path ='/kaggle/input/toledo-ohio-property-tax-delinquency/delinquent_toledo.csv'
del_hous = pd.read_csv(path)

# Rename the columns


del_hous_toledo = del_hous

del_hous_toledo.head()
del_hous_toledo.columns

del_hous_cols = {'Parcel Number':'parcel', 'Owner Name as Listed in County Records':'owner', 'Legal Description  as Listed in County Records':'description','Address as Listed in County Records' :'address', 'Total Under taking at the close of the 2023 Tax Year Collection on July 31, 2024. ':'ttax24'}

del_hous_toledo_ren = del_hous_toledo.rename(columns = del_hous_cols)

display(del_hous_toledo_ren.head())

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# Sort the DataFrame by owner and address
del_hous_toledo_ren_sorted = del_hous_toledo_ren.sort_values(by=['owner', 'address'])

location = del_hous_toledo_ren_sorted[['parcel','owner', 'address']]
address = location['address'] 
owner = location['owner']
parcel = location['parcel']

location_df = pd.DataFrame(location)
display(location_df.head())

address = location_df['address'] 
owner = location_df['owner']
parcel = location_df['parcel']
location_df.loc[:, 'city'] = 'Toledo'
location_df.loc[:, 'county'] = 'Lucas'
location_df.loc[:, 'state'] = 'Ohio'

location_df
!pip install ace-tools-open
import ace_tools_open as tools

tools.display_dataframe_to_user(name="Location Data", dataframe=location_df.head())

In [1]:
!pip install geocoder
!pip install geopy
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os

# Load the cleaned address data
df = pd.read_csv("/kaggle/input/cleaned-delinquent-tax-listing-for-2024/cleaned_toledo_addresses.csv")

import re

def split_address(addr):
    """
    Splits address into (street_number, street_name).
    E.g. '1234 Main St' → ('1234', 'Main St')
    If no number found, returns (None, addr)
    """
    if pd.isnull(addr):
        return (None, None)
    match = re.match(r'^\s*(\d+)\s+(.*)', str(addr).strip())
    if match:
        return match.groups()
    else:
        return (None, addr.strip())

# Example usage (after loading your DataFrame):
df[['street_number', 'street_name']] = df['address'].apply(
    lambda x: pd.Series(split_address(x))
)


import numpy as np

def clean_street_number(val):
    # Remove blanks and nan-like values
    if pd.isnull(val) or str(val).strip() == "" or str(val).strip().lower() == "nan":
        return np.nan
    try:
        # Convert float strings like '1311.0' or float type 1311.0 -> 1311 (int), then string
        return str(int(float(val)))
    except Exception:
        return np.nan

df["street_number"] = df["street_number"].apply(clean_street_number)
df = df.dropna(subset=["street_number", "street_name"])


   
# Fix: Convert floats to int strings (e.g., 1311.0 -> "1311"), handle NaNs
df["street_number"] = df["street_number"].fillna("").astype(float).astype("Int64").astype(str)
df["street_name"] = df["street_name"].fillna("").astype(str)
df["city"] = df["city"].fillna("").astype(str)
df["state"] = df["state"].fillna("").astype(str)

# Drop rows with missing or bad address data
df = df[~df["street_number"].str.lower().str.contains("nan")]
df = df[~df["street_name"].str.lower().str.contains("nan")]
df = df.dropna(subset=["street_number", "street_name"])

# Combine into full address
df["full_address"] = df["street_number"] + " " + df["street_name"] + ", " + df["city"] + ", " + df["state"]

# Geocoder setup (no timeout here)
geolocator = Nominatim(user_agent="toledo_delinquent_properties_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=True)

# Resume support
if os.path.exists("geocoded_partial.csv"):
    geocoded_df = pd.read_csv("geocoded_partial.csv")
    completed_addresses = set(geocoded_df["full_address"])
    print(f"Resuming: {len(completed_addresses)} already processed.")
else:
    geocoded_df = pd.DataFrame()
    completed_addresses = set()

# Correct safe_geocode with timeout passed to geocode directly
def safe_geocode(address):
    if not isinstance(address, str) or "nan" in address.lower():
        print(f"Skipping invalid address: {address}")
        return None, None
    try:
        loc = geolocator.geocode(address, timeout=15)
        if loc:
            return loc.latitude, loc.longitude
        else:
            print(f"Not found: {address}")
            return None, None
    except Exception as e:
        print(f"Error geocoding '{address}': {e}")
        time.sleep(20)
        return None, None

# Geocode in batches
batch_size = 50
new_rows = []

for idx, row in df.iterrows():
    addr = row["full_address"]
    if addr in completed_addresses:
        continue

    lat, lon = safe_geocode(addr)
    row_data = row.to_dict()
    row_data["latitude"] = lat
    row_data["longitude"] = lon
    new_rows.append(row_data)

    if len(new_rows) >= batch_size:
        new_df = pd.DataFrame(new_rows)
        geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
        geocoded_df.to_csv("geocoded_partial.csv", index=False)
        print(f"Saved {len(geocoded_df)} rows to geocoded_partial.csv")
        new_rows = []

# Final write
if new_rows:
    new_df = pd.DataFrame(new_rows)
    geocoded_df = pd.concat([geocoded_df, new_df], ignore_index=True)
    geocoded_df.to_csv("geocoded_partial.csv", index=False)
    print(f"Final save: {len(geocoded_df)} rows total.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 12.4 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.4/125.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Resuming: 4529 already processed.
Not found: 4587 MARINERS COVE DR, Toledo, Ohio
Not found: 3459 FOREST GROVE, Toledo, Ohio
Not found: 3131 MCKINNEY AVE STE L10, Toledo, Ohio
Not found: 8740 OAK HOLLOW RD, Toledo, Ohio
Not found: 1101 GRASSER ST, Toledo, Ohio
Not found: 227 CENTRAL ST, Toledo, Ohio
Not found: 227 CENTRAL ST, Toledo, Ohio
Not found: 227 CENTRAL ST, Toledo, Ohio
Not found: 2282 GARDEN CREEK DR, Toledo, Ohio
Not found: 3218 ROMAK

In [2]:
import os
print(os.listdir())

import shutil
AddreCoorEr = shutil.copy("/kaggle/working/geocoded_partial.csv")






['geocode_toledo_addresses.ipynb', 'geocoded_partial.csv', '.virtual_documents']


SameFileError: 'geocoded_partial.csv' and '/kaggle/working/geocoded_partial.csv' are the same file